In this notebook, we will work with 3D data. While our primary focus will be on utilizing the [Open3D](https://www.open3d.org/) library, we will also explore other tools like [Trimesh](https://trimesh.org/) and [Meshlab](https://www.meshlab.net/).

In section 1, we'll look at different types of data that can be found and how to load them. In section 2, a few data preprocessing techniques will be explored, including noise removal, downsampling , upsampling, and transformations. Then, we'll reconstruct a mesh from a point cloud. Finally, we'll work on a use case that you have already seen in the theory part.

In [ ]:
#@title **Imports**

%%capture
!pip install open3d trimesh --quiet

import os
import sys
import copy
import trimesh
import numpy as np
import open3d as o3d
import plotly.graph_objects as go
from IPython.display import display
from matplotlib import pyplot as plt

np.random.seed(42)

In [ ]:
# @title **Auriliary Functions**

import plotly.graph_objects as go
from plotly.subplots import make_subplots

def plot_point_clouds(pc1, pc2):
  fig = make_subplots(rows=1, cols=2,
                      specs=[[ {"type": "scatter3d"}, {"type": "scatter3d"}]])

  fig.add_trace(
      go.Scatter3d(x=pc1[:,0], y=pc1[:,1], z=pc1[:,2], mode='markers', marker=dict(size=3), name='Original Data'),
      row=1, col=1
  )

  fig.add_trace(
      go.Scatter3d(x=pc2[:,0], y=pc2[:,1], z=pc2[:,2], mode='markers', marker=dict(size=3), name='Noisy Data'),
      row=1, col=2
  )

  fig.update_layout(height=700, width=1500, showlegend=True)
  fig.show()

def plot_outliers(pc1, pc2, outliers):
    fig = make_subplots(rows=1, cols=2,
                      specs=[[ {"type": "scatter3d"}, {"type": "scatter3d"}]])

    fig.add_trace(
      go.Scatter3d(x=pc1[:,0], y=pc1[:,1], z=pc1[:,2], mode='markers', marker=dict(size=3), name='Original Data'),
      row=1, col=1
    )
    fig.add_trace(
        go.Scatter3d(x=pc2[:,0], y=pc2[:,1], z=pc2[:,2], mode='markers', marker=dict(size=3), name='Noisy Data'),
        row=1, col=2
    )
    fig.add_trace(
        go.Scatter3d(x=outliers[:,0], y=outliers[:,1], z=outliers[:,2], mode='markers', marker=dict(size=3), name='Outliers'),
        row=1, col=2
    )
    fig.update_layout(height=700, width=1500, showlegend=True)
    fig.show()

def plot_outliers2(pc1, pc2, pc3, names=['Original Data', 'Noisy Data', 'Filtered Data']):
    fig = make_subplots(rows=1, cols=3,
                      specs=[[{"type": "scatter3d"},{"type": "scatter3d"},{"type": "scatter3d"}]])

    fig.add_trace(
      go.Scatter3d(x=pc1[:,0], y=pc1[:,1], z=pc1[:,2], mode='markers', marker=dict(size=3), name=names[0]),
      row=1, col=1
    )
    fig.add_trace(
        go.Scatter3d(x=pc2[:,0], y=pc2[:,1], z=pc2[:,2], mode='markers', marker=dict(size=3), name=names[1]),
        row=1, col=2
    )
    fig.add_trace(
        go.Scatter3d(x=pc3[:,0], y=pc3[:,1], z=pc3[:,2], mode='markers', marker=dict(size=3), name=names[2]),
        row=1, col=3
    )
    fig.update_layout(height=700, width=1500, showlegend=True)
    fig.show()

# **1. Types of 3D data**

## 1.1. Point Cloud

Point clouds consist of discrete 3D points irregularly
sampled from continuous surfaces. Each point is defined by its x, y, and z coordinates.

In [ ]:
# Load point cloud with open3d (Stanford bunny)
bunny = o3d.data.BunnyMesh()
bunny = o3d.io.read_point_cloud(bunny.path)

# Get vertices with open3d
vertices_o3d = np.asarray(bunny.points)
print('Shape:', vertices_o3d.shape)

# Save with open3d
o3d.io.write_point_cloud('bunny.ply', bunny, write_ascii=True)

# Save with numpy
np.savetxt('bunny.xyz', vertices_o3d, delimiter=' ', fmt='%f')

# Visualize the point cloud

# Local
# o3d.visualization.draw_geometries([bunny])

# Colab
o3d.visualization.draw_plotly([bunny])

In [ ]:
# Load point cloud with trimesh
bunny = trimesh.load('bunny.ply')

# Get vertices with trimesh
vertices_trimesh = bunny.vertices

# Change point colors
bunny.visual.vertex_colors = [0, 0, 255, 255]

# Visualize the point cloud
scene = trimesh.Scene()
scene.add_geometry(bunny)
scene.show()

## 1.2. Mesh

3D object representation consisting of a collection of vertices and polygons.

In [ ]:
# Load Mesh
bunny = o3d.data.BunnyMesh()
bunny = o3d.io.read_triangle_mesh(bunny.path)
o3d.visualization.draw_plotly([bunny])

# Get vertices and faces
vertices = np.array(bunny.vertices)
faces = np.array(bunny.triangles)

# Save
o3d.io.write_triangle_mesh('bunny.obj', bunny)

In [ ]:
bunny = trimesh.load_mesh('bunny.obj', process=False)

# Get vertices and faces
vertices = bunny.vertices
faces = bunny.faces

bunny.show()

## 1.3. Octree

Octree is a tree data structure in which each internal node has exactly eight children.

1. Given a set of points we put them into a cuboid of dimension:
    \begin{equation}
      (y_{max}, y_{min}) \cdot (x_{max}, x_{min}) \cdot (z_{max}, z_{min})
    \end{equation}
2. If the cuboid has more than a certain number of points, we divide it into 8 smaller cuboids
3. We repeat the process until we have a certain number of points in each cuboid

In [ ]:
bunny = o3d.io.read_point_cloud('bunny.ply')
octree = o3d.geometry.Octree(max_depth=6)
octree.convert_from_point_cloud(bunny, size_expand=0.01)

# Local
#o3d.visualization.draw_geometries([octree])

<center><img src="https://drive.google.com/uc?export=view&id=1-5I8QsvFvgGJJAk_2BQ5I-u9CxFZbIUX" style="float:left; padding:0.7em" width=900/></center>

## 1.4. Voxel Grid

3D cube located on a three-dimensional grid

In [ ]:
bunny = o3d.data.BunnyMesh()
bunny = o3d.io.read_triangle_mesh(bunny.path)
voxel_grid = o3d.geometry.VoxelGrid.create_from_triangle_mesh(bunny, voxel_size=0.01)

voxels = voxel_grid.get_voxels()
x, y, z = [], [], []
for voxel in voxels:
    x.append(voxel.grid_index[0])
    y.append(voxel.grid_index[1])
    z.append(voxel.grid_index[2])

fig = go.Figure(data=go.Scatter3d(x=x, y=y, z=z, mode='markers', marker=dict(size=5)))
fig.show()

# **2. Preprocessing Techniques**

- Ground truth \
https://drive.google.com/file/d/1ORt1SnoU94y9CMg8BcAEJk0QmZmoNRjc/view?usp=sharing

- With noise \
https://drive.google.com/file/d/1TBS9YnK4XbeD2IjGaROK_y8PRkyt_Sah/view?usp=sharing

In [ ]:
%%capture
!gdown 1ORt1SnoU94y9CMg8BcAEJk0QmZmoNRjc
!gdown 1TBS9YnK4XbeD2IjGaROK_y8PRkyt_Sah

In [ ]:
# Read chair.off and chair.xyz

chair = o3d.io.read_triangle_mesh('chair.off')
chair_noise = o3d.io.read_point_cloud('chair.xyz')

chair_vertices = np.array(chair.vertices)
chair_noise_vertices = np.array(chair_noise.points)

plot_point_clouds(chair_vertices, chair_noise_vertices)

## 2.1. Outlier Removal

### Stadistical outlier removal

`remove_statistical_outlier`: Removes points that are further away from their neighbors. For each point the mean distance from it to all its neighbors is computed. Then, if the mean distance of the point is outside an interval defined by the global distances mean and standard deviation then the point is an outlier.

`Parameters`
- nb_neighbors: Number of neighbors around the target point.
- std_ratio: Standard deviation ratio.

`Output`
- Filtered point cloud and the indices of the input point cloud that are not outliers

`Example:`
- input_point_cloud, idx = point_cloud.remove_statistical_outlier(nb_neighbors=20, std_ratio=0.01)

---

To select vertices in a point cloud we are going to use the function `select_by_index`.

`Parameters`
- indices: Indices of points to be selected.
- invert : Set to True to invert the selection of indices.

`Example:`
- point_cloud_without_outliers = point_cloud.select_by_index(idx, invert=False)
- outliers = point_cloud.select_by_index(idx, invert=True)

> ### **Exercise:** Remove outliers from the chair point cloud by using the `remove_statistical_outlier` function.

In [ ]:
# To-do

In [ ]:
plot_outliers(chair_vertices, result_vertices, outliers_vertices)
# plot_outliers2(chair_vertices, chair_noise_vertices, result_vertices)

### Density-Based Spatial Clustering of Applications with Noise (DBSCAN) clustering

`DBSCAN`: automatically detect the optimal number of clusters based on the density of the data points.

`Parameters`
- eps: The maximum distance between two samples for one to be considered as in the neighborhood of the other.
- min_samples: Minimum number of points to form a cluster.

`Output`
- Cluster labels

`Example:`
- clusters = DBSCAN(eps=0.01, min_samples=10).fit(vertices)
- outliers = vertices[clusters.labels_ == -1]

> ### **Exercise:** Remove outliers from the chair point cloud by using the DBSCAN.

In [ ]:
from sklearn.cluster import DBSCAN

# To-do

In [ ]:
plot_outliers(chair_vertices, chair_noise_vertices, result)
# plot_outliers2(chair_vertices, chair_noise_vertices, result)

## 2.2 Downsampling / Upsampling

### Downsampling



Downsampling a point cloud is helpful for reducing the computational and memory resources required for subsequent processing steps.

`voxel_down_sample`:

`Parameters`
- voxel_size: Voxel size to downsample into.

`Output`
- Point Cloud

---

`uniform_down_sample`:

`Parameters`
- every_k_points: Sample rate

`Output`
- Point Cloud

In [ ]:
bunny = o3d.io.read_point_cloud('bunny.ply')
bunny_vertices = np.array(bunny.points)

d_voxel = bunny.voxel_down_sample(voxel_size=0.01)
d_uniform = bunny.uniform_down_sample(every_k_points=100)

d_voxel_vertices = np.array(d_voxel.points)
d_uniform_vertices = np.array(d_uniform.points)

plot_outliers2(bunny_vertices, d_voxel_vertices, d_uniform_vertices, names=['Original', 'Voxel', 'Uniform'])

### Upsampling

> #### **Exercise:** Upsample a downsampled point cloud using the KNN algorithm.  For every point in the cloud, identify its k nearest neighbors using the `NearestNeighbors` module from `sklearn.neighbors`. Then, generate a new point by averaging the coordinates of its neighbors. Finally, display the upsampled point cloud.

In [ ]:
from sklearn.neighbors import NearestNeighbors
def upsample_point_cloud_with_knn(original_vertices):
  # To-Do



## 2.3. Transformations

### Translations, Rotations and Scale

In [ ]:
chair = o3d.io.read_point_cloud('chair.xyz')
chair_test = copy.deepcopy(chair)
chair_test.paint_uniform_color([0, 0, 1])

# Translations
translation = [0.5, 0.5, 0.5]
chair_test.translate(translation)

# Rotation (Euler angles)
rotation = [0, 0, np.pi / 4]
chair_test.rotate(chair_test.get_rotation_matrix_from_xyz(rotation))

# Scale
scale=0.2
chair_test.scale(scale, center=chair_test.get_center())

In [ ]:
o3d.visualization.draw_plotly([chair, chair_test])

### Transformation matrix

The transformation matrix is generated from a combination of translation, rotations, and scalings.


- The translation matrix:

\begin{equation}
M=\begin{bmatrix}
1 & 0 & 0 & t_{x} \\
0 & 1 & 0 & t_{y} \\
0 & 0 & 1 & t_{z} \\
0 & 0 & 0 & 1
\end{bmatrix}
\end{equation}

- The scaling matrix:

\begin{equation}
M=\begin{bmatrix}
s_{x} & 0 & 0 & 0 \\
0 & s_{y} & 0 & 0 \\
0 & 0 & s_{z} & 0 \\
0 & 0 & 0 & 1
\end{bmatrix}
\end{equation}

- The rotations are represented by the matrices $R_x(\theta_x), R_y(\theta_y), R_z(\theta_z)$ respectively, where
$\theta$ is the angle of rotation. The matrices are defined as follows:

\begin{equation}
R_x(\theta) = \begin{bmatrix}
1 & 0 & 0 & 0 \\
0 & \cos(\theta) & -\sin(\theta) & 0 \\
0 & \sin(\theta) & \cos(\theta) & 0 \\
0 & 0 & 0 & 1
\end{bmatrix}
\end{equation}

\begin{equation}
R_y(\theta) = \begin{bmatrix}
\cos(\theta) & 0 & \sin(\theta) & 0 \\
0 & 1 & 0 & 0 \\
-\sin(\theta) & 0 & \cos(\theta) & 0 \\
0 & 0 & 0 & 1
\end{bmatrix}
\end{equation}

\begin{equation}
R_z(\theta) = \begin{bmatrix}
\cos(\theta) & -\sin(\theta) & 0 & 0 \\
\sin(\theta) & \cos(\theta) & 0 & 0 \\
0 & 0 & 1 & 0 \\
0 & 0 & 0 & 1
\end{bmatrix}
\end{equation}



The final transformation matrix $M$ that combines translation, scaling, and rotation can be constructed by multiplying the individual transformation matrices. The order of multiplication is important and depends on the desired sequence of transformations. For example, if one wishes to first scale, then rotate, and finally translate an object, the combined transformation matrix $M$ would be calculated as:

\begin{equation}
M = T \cdot R_z(\theta_z) \cdot R_y(\theta_y) \cdot R_x(\theta_x) \cdot S
\end{equation}

> ### **Exercise:** Create a transformation matrix using specified translation, rotations, and scale parameters. Once created, apply this matrix to the chair point cloud.
>
> - Parameters:
>   - Traslation =  [0.5, 0.5, 0.5]
>   - Rotation =  [180, 45, 45]
>   - Scale =  [1.5, 1.5, 1.5]
>
> - Hint: To create the rotation component of the matrix, you'll need to convert angles to radians.

In [ ]:
# Create scaling matrix


# Create rotation matrices


# Create translation matrix


# Combine the matrices to form M
# M = T.dot(Rz). ...

# Apply the transformation matrix (use the transform function)

# Visualize

# **3. Generate Mesh**

In [ ]:
%%capture
!gdown 1g5deSg90J6kpJfrbwMmdiardH5-QQ2fg

### Ball pivoting

In [ ]:
chair = o3d.io.read_point_cloud('chair_50000.xyz')
chair_vertices = np.array(chair.points)
chair.estimate_normals()

In [ ]:
o3d.io.write_point_cloud("chair_pc.ply", chair)

In [ ]:
chair_mesh = o3d.geometry.TriangleMesh.create_from_point_cloud_ball_pivoting(chair, o3d.utility.DoubleVector([0.02]))
o3d.visualization.draw_plotly([chair_mesh])

### Poisson recostruction

In [ ]:
chair_mesh, _ = o3d.geometry.TriangleMesh.create_from_point_cloud_poisson(chair, depth=8)
o3d.visualization.draw_plotly([chair_mesh])

# **4. Exercise: Use case**

### Pipeline

1. Load Point Clouds
2. Preprocess Point Clouds
3. Read the Calibration File: [configparser](https://docs.python.org/3/library/configparser.html)
3. Apply the Calibration Matrix
4. Merge Point Clouds
5. Mesh Generation
6. Visualize

<center><img src="https://drive.google.com/uc?export=view&id=1bLFR-v-3cFr0XFxFHWNO_6QwWVQBFAQk" style="float:left; padding:0.7em" width=900/></center>

In [ ]:
#@title Download Data
%%capture
!gdown https://drive.google.com/drive/folders/1d4FaXXwmoytc_1GRn8_GROvQNYwMvApH?usp=sharing -O t4d_data --folder

In [ ]:
# To-Do